# Kavra stüdyosu (Kaggle, GPU)

Kavra'nın **tamamını** (arayüz + video render + XTTS v2/Anka/Piper, istersen Chatterbox) Kaggle GPU'sunda kurar
ve cloudflared ile internete açar.

1. Sağ paneldeki **Session options**'da: **Accelerator → GPU T4 x2** seç (**P100 seçme**: projedeki CUDA 12.8 Torch
   Pascal kartları desteklemez) ve **Internet → On** yap (telefon doğrulaması ister).
2. Hücreleri sırayla çalıştır. İlk kurulum ~10-15 dk sürer; aynı oturumda tekrar çalıştırınca atlanır.
3. 2. hücrenin yazdırdığı **bağlantıyı** aç: arayüz şifreyle korunur, bağlantı şifreyi içerir — kimseyle paylaşma.
4. Videonu üret, indir; bitince 4. hücreyi çalıştır ve oturumu **Stop session** ile kapat (GPU kotan boşa gitmez).

**İsteğe bağlı (Add-ons → Secrets, not defterine bağla):**
- `KAVRA_ERISIM_SIFRESI` — her oturumda aynı kalan şifre (en az 16 karakter). Yoksa her seferinde rastgele üretilir.
- `GEMINI_API_KEY`, `OPENAI_API_KEY`, `ELEVENLABS_API_KEY` — anlatı/seslendirme anahtarları (arayüzden de girilebilir).

> Ses ayarlarında *Çalıştırma yeri* **Bu bilgisayar** kalmalı: burada "bu bilgisayar" Kaggle'ın GPU'su.
> Projeler `/kaggle/working/kavra_data` altında durur; oturum kapanınca silinir — videonu kapatmadan önce indir.

In [ ]:
# 1) Depoyu GitHub'dan çek ve kur
REPO_URL = "https://github.com/davutsimsekk/Kavra.git"
BRANCH = "main"
CHATTERBOX_KUR = False

import os, subprocess, sys

REPO = "/root/kavra/Kavra"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "YOK - Session options > Accelerator > GPU T4 x2 seç")
if "P100" in gpu:
    print("UYARI: P100 bu Torch sürümünde desteklenmez; XTTS çalışmaz. Accelerator'ı GPU T4 x2 yap.")

os.makedirs(os.path.dirname(REPO), exist_ok=True)
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
import importlib
from tools import colab_studio
colab_studio = importlib.reload(colab_studio)  # hücre yeniden çalışınca çekilen yeni kod kullanılsın
assert colab_studio.KAGGLE, "Bu not defteri Kaggle içindir; Colab'da Kavra_Studyo_Colab.ipynb'yi aç."
colab_studio.install(chatterbox=CHATTERBOX_KUR)

In [ ]:
# 2) Kavra'yı başlat ve internete aç
studio = colab_studio.start()

In [ ]:
# 3) Açık tut: sunucuyu ve tüneli izler, düşerse yeniden başlatır (■ ile izlemeyi bırakabilirsin)
studio.watch()

In [ ]:
# 4) Bitti mi? Önce 3. hücreyi ■ ile durdur, sonra bunu çalıştır; ardından sağ üstten Stop session.
# İndirmediğin videolar oturum kapanınca silinir!
studio.stop()